# Regression, gradient descent, and a small neural-network demo

This notebook demonstrates core regression ideas on synthetic data, then implements a tiny linear regressor trained with gradient descent. A short, optional PyTorch section shows how the same ideas extend to multi-dimensional data.

Reusable code (SimpleLinearRegression) has been extracted into src/ so it can be imported and reused across projects. Plots and data generation use deterministic seeds for reproducibility.

### Part 1: Linear regression on simple synthetic datasets

We begin with straightforward linear regression on basic synthetic datasets and build up from there.

We will generate three small datasets and visualize them:

- linear: y = a0 + a1 x + noise
- convex: a 2nd/3rd-order polynomial with noise
- trigonometric: y = wc cos(x) + ws sin(x) + noise

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Reproducibility for all synthetic data in this notebook
np.random.seed(42)


def generate_polynomial_dataset(weights, n_samples=100, noise_level=0.1):
    """Generate a polynomial dataset with added Gaussian noise."""
    X = np.linspace(-3, 3, n_samples).reshape(-1, 1)
    y = np.zeros(n_samples)
    for i, weight in enumerate(weights):
        y += weight * (X**i).flatten()
    y += np.random.normal(0, noise_level, size=n_samples)
    return X, y


def generate_trigonometric_dataset(w_c, w_s, n_samples=100, noise_level=0.1):
    """Generate a trigonometric dataset with added Gaussian noise."""
    X = np.linspace(-2 * np.pi, 2 * np.pi, n_samples).reshape(-1, 1)
    y = w_c * np.cos(X).flatten() + w_s * np.sin(X).flatten()
    y += np.random.normal(0, noise_level, size=n_samples)
    return X, y


datasets = {
    "linear": generate_polynomial_dataset([1, 2], 50, 0.5),
    "convex": generate_polynomial_dataset([1, 2, 3], 50, 2.5),
    "trigonometric": generate_trigonometric_dataset(2, -1, 50, 0.5),
}

# Visualize datasets
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, dataset in zip(axes, datasets.items()):
    title, (X, y) = dataset
    ax.scatter(X, y, alpha=0.5)
    ax.set_title(title)
    ax.set_xlabel("Feature")
    ax.set_ylabel("Target")
plt.tight_layout()
plt.show()

The three datasets show different relationships: a clearly linear pattern, a curved polynomial trend, and a periodic pattern. A single straight line cannot capture curvature or periodicity without transforming the features.

##### Train/test split

Split each dataset into training and testing sets to evaluate generalization.

In [ ]:
from sklearn.model_selection import train_test_split

# Unpack datasets
X_linear, Y_linear = datasets["linear"]
X_convex, Y_convex = datasets["convex"]
X_tri, Y_tri = datasets["trigonometric"]

random_state = 42
np.random.seed(random_state)

# Create splits
X_linear_train, X_linear_test, Y_linear_train, Y_linear_test = train_test_split(
    X_linear, Y_linear, test_size=0.2, random_state=random_state
)
X_convex_train, X_convex_test, Y_convex_train, Y_convex_test = train_test_split(
    X_convex, Y_convex, test_size=0.2, random_state=random_state
)
X_tri_train, X_tri_test, Y_tri_train, Y_tri_test = train_test_split(
    X_tri, Y_tri, test_size=0.2, random_state=random_state
)

Visualize the train/test splits.

In [ ]:
# Create the subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot the linear dataset
axes[0].scatter(X_linear_train, Y_linear_train, color="blue", label="Train", alpha=0.5)
axes[0].scatter(X_linear_test, Y_linear_test, color="red", label="Test", alpha=0.5)
axes[0].set_title("Linear Relationship")
axes[0].set_xlabel("Feature")
axes[0].set_ylabel("Target")
axes[0].legend()

# Plot the convex dataset
axes[1].scatter(X_convex_train, Y_convex_train, color="blue", label="Train", alpha=0.5)
axes[1].scatter(X_convex_test, Y_convex_test, color="red", label="Test", alpha=0.5)
axes[1].set_title("Convex Relationship")
axes[1].set_xlabel("Feature")
axes[1].legend()

# Plot the trigonometric dataset
axes[2].scatter(X_tri_train, Y_tri_train, color="blue", label="Train", alpha=0.5)
axes[2].scatter(X_tri_test, Y_tri_test, color="red", label="Test", alpha=0.5)
axes[2].set_title("Trigonometric Relationship")
axes[2].set_xlabel("Feature")
axes[2].legend()

plt.tight_layout()
plt.show()

##### Baseline linear models

Fit a LinearRegression model to each dataset. This provides a baseline and highlights where a straight line is insufficient.

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model_linear = LinearRegression()
linear_model_convex = LinearRegression()
linear_model_trigonometric = LinearRegression()

linear_model_linear.fit(X_linear_train, Y_linear_train)
linear_model_convex.fit(X_convex_train, Y_convex_train)
linear_model_trigonometric.fit(X_tri_train, Y_tri_train)

Y_linear_train_pred = linear_model_linear.predict(X_linear_train)
Y_convex_train_pred = linear_model_convex.predict(X_convex_train)
Y_tri_train_pred = linear_model_trigonometric.predict(X_tri_train)

Y_linear_test_pred = linear_model_linear.predict(X_linear_test)
Y_convex_test_pred = linear_model_convex.predict(X_convex_test)
Y_tri_test_pred = linear_model_trigonometric.predict(X_tri_test)

Below, we plot data and fitted lines using the training predictions. We'll evaluate with the test predictions next.

In [ ]:
# Plot the results along with the regression lines for each dataset
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Linear dataset and model
axes[0].scatter(X_linear_train, Y_linear_train, color="blue", label="Train")
axes[0].scatter(X_linear_test, Y_linear_test, color="orange", label="Test")
axes[0].plot(
    X_linear_train, Y_linear_train_pred, color="green", label="Linear Regression"
)
axes[0].set_title("Linear Dataset Regression")
axes[0].legend()

# Convex dataset and model
axes[1].scatter(X_convex_train, Y_convex_train, color="blue", label="Train")
axes[1].scatter(X_convex_test, Y_convex_test, color="orange", label="Test")
axes[1].plot(
    X_convex_train, Y_convex_train_pred, color="green", label="Linear Regression"
)
axes[1].set_title("Convex Dataset Regression")
axes[1].legend()

# Trigonometric dataset and model
axes[2].scatter(X_tri_train, Y_tri_train, color="blue", label="Train")
axes[2].scatter(X_tri_test, Y_tri_test, color="orange", label="Test")
axes[2].plot(X_tri_train, Y_tri_train_pred, color="green", label="Linear Regression")
axes[2].set_title("Trigonometric Dataset Regression")
axes[2].legend()

plt.tight_layout()
plt.show()

##### Evaluation: Mean Squared Error (MSE) and R-squared

Compute MSE and R^2 for both training and test sets.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

mse_linear_train = mean_squared_error(Y_linear_train, Y_linear_train_pred)
mse_linear_test = mean_squared_error(Y_linear_test, Y_linear_test_pred)
r2_linear_train = r2_score(Y_linear_train, Y_linear_train_pred)
r2_linear_test = r2_score(Y_linear_test, Y_linear_test_pred)

mse_convex_train = mean_squared_error(Y_convex_train, Y_convex_train_pred)
mse_convex_test = mean_squared_error(Y_convex_test, Y_convex_test_pred)
r2_convex_train = r2_score(Y_convex_train, Y_convex_train_pred)
r2_convex_test = r2_score(Y_convex_test, Y_convex_test_pred)

mse_tri_train = mean_squared_error(Y_tri_train, Y_tri_train_pred)
mse_tri_test = mean_squared_error(Y_tri_test, Y_tri_test_pred)
r2_tri_train = r2_score(Y_tri_train, Y_tri_train_pred)
r2_tri_test = r2_score(Y_tri_test, Y_tri_test_pred)

In [ ]:
# Organize the results into a structured format
results = {
    "Linear": {
        "MSE Train": mse_linear_train,
        "MSE Test": mse_linear_test,
        "R2 Train": r2_linear_train,
        "R2 Test": r2_linear_test,
    },
    "Convex": {
        "MSE Train": mse_convex_train,
        "MSE Test": mse_convex_test,
        "R2 Train": r2_convex_train,
        "R2 Test": r2_convex_test,
    },
    "Trigonometric": {
        "MSE Train": mse_tri_train,
        "MSE Test": mse_tri_test,
        "R2 Train": r2_tri_train,
        "R2 Test": r2_tri_test,
    },
}

# Print the results
for dataset, metrics in results.items():
    print(f"{dataset} Dataset:")
    print(f"  MSE Train: {metrics['MSE Train']:.4f}")
    print(f"  MSE Test: {metrics['MSE Test']:.4f}")
    print(f"  R2 Train: {metrics['R2 Train']:.4f}")
    print(f"  R2 Test: {metrics['R2 Test']:.4f}")
    print()  # Add a blank line for better readability

##### Notes on the results

Briefly interpret the baseline metrics and what R^2 represents.

Observations

1. Linear dataset: A linear model typically performs well here, with low MSE and high R^2 on both train and test.
2. Convex dataset: A straight line underfits a curved relationship, often yielding higher MSE and lower R^2.
3. Trigonometric dataset: A straight line also underfits periodic behavior, so errors are larger and R^2 may be low or negative.

R^2 (coefficient of determination) measures the proportion of variance explained by the model. R^2 = 0 means the model explains none of the variance beyond a constant baseline; R^2 = 1 indicates a perfect fit on the evaluated data.

##### Polynomial features

We can map x to a higher-dimensional polynomial feature space where a non-linear relationship becomes linear in the transformed variables.

If a relationship is non-linear in x, we can expand x with PolynomialFeatures so that a linear model fits the transformed features. For example, y = x^2 is linear in the feature α = x^2. We will use sklearn's make_pipeline and PolynomialFeatures.

We now fit polynomial regressors for the linear, convex, and trigonometric datasets and evaluate on both train and test splits.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Define degrees for polynomial features
degree_linear = 1  # For the linear dataset, a degree of 1 is just a linear relationship
degree_convex = 3  # For the convex dataset, let's try a third-degree (cubic) model
degree_tri = 10  # For the trigonometric dataset, a higher degree might capture the sine/cosine waves

# Linear model
linear_model = make_pipeline(PolynomialFeatures(degree_linear), LinearRegression())
linear_model.fit(X_linear_train, Y_linear_train)
Y_linear_train_pred_poly = linear_model.predict(X_linear_train)
Y_linear_test_pred_poly = linear_model.predict(X_linear_test)

# Convex model
convex_model = make_pipeline(PolynomialFeatures(degree_convex), LinearRegression())
convex_model.fit(X_convex_train, Y_convex_train)
Y_convex_train_pred_poly = convex_model.predict(X_convex_train)
Y_convex_test_pred_poly = convex_model.predict(X_convex_test)

# Trigonometric model
tri_model = make_pipeline(PolynomialFeatures(degree_tri), LinearRegression())
tri_model.fit(X_tri_train, Y_tri_train)
Y_tri_train_pred_poly = tri_model.predict(X_tri_train)
Y_tri_test_pred_poly = tri_model.predict(X_tri_test)

In [ ]:
# Plot the results along with the regression lines for each dataset
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Prepare sorted X values for plotting polynomial regression lines
X_linear_train_sorted_idx = np.argsort(X_linear_train.ravel())
X_convex_train_sorted_idx = np.argsort(X_convex_train.ravel())
X_tri_train_sorted_idx = np.argsort(X_tri_train.ravel())

# Linear dataset and model
axes[0].scatter(X_linear_train, Y_linear_train, color="blue", label="Train")
axes[0].scatter(X_linear_test, Y_linear_test, color="orange", label="Test")
axes[0].plot(
    X_linear_train[X_linear_train_sorted_idx],
    Y_linear_train_pred_poly[X_linear_train_sorted_idx],
    color="green",
    label="Linear Regression",
)
axes[0].set_title("Linear Dataset Regression")
axes[0].legend()

# Convex dataset and model
axes[1].scatter(X_convex_train, Y_convex_train, color="blue", label="Train")
axes[1].scatter(X_convex_test, Y_convex_test, color="orange", label="Test")
axes[1].plot(
    X_convex_train[X_convex_train_sorted_idx],
    Y_convex_train_pred_poly[X_convex_train_sorted_idx],
    color="green",
    label="Linear Regression",
)
axes[1].set_title("Convex Dataset Regression")
axes[1].legend()

# Trigonometric dataset and model
axes[2].scatter(X_tri_train, Y_tri_train, color="blue", label="Train")
axes[2].scatter(X_tri_test, Y_tri_test, color="orange", label="Test")
axes[2].plot(
    X_tri_train[X_tri_train_sorted_idx],
    Y_tri_train_pred_poly[X_tri_train_sorted_idx],
    color="green",
    label="Linear Regression",
)
axes[2].set_title("Trigonometric Dataset Regression")
axes[2].legend()

plt.tight_layout()
plt.show()

Compute MSE and R^2 for training and testing sets under polynomial features.

In [ ]:
mse_linear_test_poly = mean_squared_error(Y_linear_test, Y_linear_test_pred_poly)
r2_linear_test_poly = r2_score(Y_linear_test, Y_linear_test_pred_poly)

mse_convex_test_poly = mean_squared_error(Y_convex_test, Y_convex_test_pred_poly)
r2_convex_test_poly = r2_score(Y_convex_test, Y_convex_test_pred_poly)

mse_tri_test_poly = mean_squared_error(Y_tri_test, Y_tri_test_pred_poly)
r2_tri_test_poly = r2_score(Y_tri_test, Y_tri_test_pred_poly)

mse_linear_train_poly = mean_squared_error(Y_linear_train, Y_linear_train_pred_poly)
r2_linear_train_poly = r2_score(Y_linear_train, Y_linear_train_pred_poly)

mse_convex_train_poly = mean_squared_error(Y_convex_train, Y_convex_train_pred_poly)
r2_convex_train_poly = r2_score(Y_convex_train, Y_convex_train_pred_poly)

mse_tri_train_poly = mean_squared_error(Y_tri_train, Y_tri_train_pred_poly)
r2_tri_train_poly = r2_score(Y_tri_train, Y_tri_train_pred_poly)

In [ ]:
# Print the MSE and R^2 values for both training and testing sets for each dataset with polynomial features
print("Linear Dataset with Polynomial Features:")
print(f"Train - MSE: {mse_linear_train_poly:.4f}, R^2: {r2_linear_train_poly:.4f}")
print(f"Test - MSE: {mse_linear_test_poly:.4f}, R^2: {r2_linear_test_poly:.4f}\n")

print("Convex Dataset with Polynomial Features:")
print(f"Train - MSE: {mse_convex_train_poly:.4f}, R^2: {r2_convex_train_poly:.4f}")
print(f"Test - MSE: {mse_convex_test_poly:.4f}, R^2: {r2_convex_test_poly:.4f}\n")

print("Trigonometric Dataset with Polynomial Features:")
print(f"Train - MSE: {mse_tri_train_poly:.4f}, R^2: {r2_tri_train_poly:.4f}")
print(f"Test - MSE: {mse_tri_test_poly:.4f}, R^2: {r2_tri_test_poly:.4f}\n")

##### Model capacity, underfitting, and overfitting

Higher-degree polynomials can fit complex patterns but may overfit. Underfitting occurs when the model is too simple (high bias); overfitting occurs when it captures noise (high variance).

Typical behavior to expect

- Linear dataset: Degree 1 should already fit well; increasing degree may not help and can hurt test performance due to overfitting.
- Convex dataset: A moderate degree (e.g., 2 or 3) often captures curvature well.
- Trigonometric dataset: Very high-degree polynomials can approximate periodic signals on the training interval but may overfit and generalize poorly.

Always validate with held-out data to detect over/underfitting.

##### Regularization: Ridge, Lasso, and ElasticNet

Regularization adds a penalty to the loss to discourage overly large coefficients and reduce overfitting:
- Ridge (L2) shrinks coefficients toward zero.
- Lasso (L1) can drive some coefficients exactly to zero (sparse solutions).
- ElasticNet combines L1 and L2.

We will compare these on a high-degree polynomial fit to the trigonometric dataset.

We compare plain polynomial regression against Lasso, Ridge, and ElasticNet on the trigonometric data to see how regularization affects the fitted curve and generalization.

In [ ]:
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Use a higher degree to highlight regularization effects on a wavy target
degree = 20
alpha_value = 0.01  # regularization strength

model_default = make_pipeline(PolynomialFeatures(degree), LinearRegression())
model_lasso = make_pipeline(PolynomialFeatures(degree), Lasso(alpha=alpha_value))
model_ridge = make_pipeline(PolynomialFeatures(degree), Ridge(alpha=alpha_value))
model_elastic = make_pipeline(
    PolynomialFeatures(degree), ElasticNet(alpha=alpha_value, l1_ratio=0.5)
)

In [ ]:
model_default.fit(X_tri_train, Y_tri_train)
model_lasso.fit(X_tri_train, Y_tri_train)
model_ridge.fit(X_tri_train, Y_tri_train)
model_elastic.fit(X_tri_train, Y_tri_train)

X_tri_dense = np.linspace(X_tri_train.min(), X_tri_train.max(), 400).reshape(-1, 1)

Y_tri_dense_pred_default = model_default.predict(X_tri_dense)
Y_tri_dense_pred_lasso = model_lasso.predict(X_tri_dense)
Y_tri_dense_pred_ridge = model_ridge.predict(X_tri_dense)
Y_tri_dense_pred_elastic = model_elastic.predict(X_tri_dense)

plt.figure(figsize=(14, 10))

plt.subplot(2, 2, 1)
plt.scatter(X_tri_train, Y_tri_train, color="blue", label="Train", alpha=0.5)
plt.scatter(X_tri_test, Y_tri_test, color="orange", label="Test", alpha=0.5)
plt.plot(
    X_tri_dense,
    Y_tri_dense_pred_default,
    color="green",
    label="Default Linear Regression",
)
plt.title("Trigonometric Dataset with Default Linear Regression")
plt.xlabel("Feature")
plt.ylabel("Target")
plt.legend()

plt.subplot(2, 2, 2)
plt.scatter(X_tri_train, Y_tri_train, color="blue", label="Train", alpha=0.5)
plt.scatter(X_tri_test, Y_tri_test, color="orange", label="Test", alpha=0.5)
plt.plot(X_tri_dense, Y_tri_dense_pred_lasso, color="green", label="Lasso Regression")
plt.title("Trigonometric Dataset with Lasso Regression")
plt.xlabel("Feature")
plt.ylabel("Target")
plt.legend()

plt.subplot(2, 2, 3)
plt.scatter(X_tri_train, Y_tri_train, color="blue", label="Train", alpha=0.5)
plt.scatter(X_tri_test, Y_tri_test, color="orange", label="Test", alpha=0.5)
plt.plot(X_tri_dense, Y_tri_dense_pred_ridge, color="green", label="Ridge Regression")
plt.title("Trigonometric Dataset with Ridge Regression")
plt.xlabel("Feature")
plt.ylabel("Target")
plt.legend()

plt.subplot(2, 2, 4)
plt.scatter(X_tri_train, Y_tri_train, color="blue", label="Train", alpha=0.5)
plt.scatter(X_tri_test, Y_tri_test, color="orange", label="Test", alpha=0.5)
plt.plot(
    X_tri_dense, Y_tri_dense_pred_elastic, color="green", label="ElasticNet Regression"
)
plt.title("Trigonometric Dataset with ElasticNet Regression")
plt.xlabel("Feature")
plt.ylabel("Target")
plt.legend()

plt.tight_layout()
plt.show()

### Part 2: Implementing linear regression via gradient descent

We now implement a minimal linear regressor trained with gradient descent and use it for a simple 1D example.

The mean-squared error (MSE) is a common loss for regression.

$$L_\text{MSE} = \frac{1}{n}\sum_{i=1}^n(\hat{Y}_i - Y_i)^2$$
$$\hat{Y}_i = \theta_1 X_i + \theta_2$$

##### Gradients of MSE for the line ŷ = θ1 x + θ2

We derive ∂L/∂θ1 and ∂L/∂θ2 for use in gradient descent.

∂L/∂θ1 = (1/n) ∑ 2 (θ1 X_i + θ2 − Y_i) X_i = (1/n) ∑ 2 (Ŷ_i − Y_i) X_i

∂L/∂θ2 = (1/n) ∑ 2 (θ1 X_i + θ2 − Y_i) = (1/n) ∑ 2 (Ŷ_i − Y_i)

##### A reusable SimpleLinearRegression class

The gradient-descent implementation has been extracted to src/simple_linear_regression.py so it can be imported and reused.

We will use gradient descent to iteratively update the slope and intercept until convergence. See the src/ module for the complete implementation.

In [ ]:
# Make the local project src/ importable from this notebooks/ folder
import os, sys
project_root = os.path.abspath(os.path.join(".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
# Import the reusable class from the local src/ package
from src.simple_linear_regression import SimpleLinearRegression

A quick smoke test for the implementation using simple synthetic pairs. This is not meant to be exhaustive; it just sanity-checks behavior on clearly linear and clearly non-linear relationships.

In [ ]:
def test_linear_regression(model, X, Y):
    model.fit(X, Y)
    predictions = model.predict(X)
    relative_error = np.mean(np.abs((Y - predictions) / (Y + 1e-8)))
    return relative_error < 0.02

# Simple test datasets
test_datasets = [
    (np.array([1, 2, 3, 4, 5]), np.array([2, 4, 6, 8, 10])),  # linear
    (np.array([1, 2, 3, 4, 5]), np.array([1, 4, 9, 16, 25])),  # quadratic
    (np.array([0, 0, 1, 1]), np.array([0, 1, 1, 0])),          # XOR-like
]

# Evaluate a fresh model per dataset
results = []
for X, Y in test_datasets:
    model = SimpleLinearRegression(learning_rate=0.001, iterations=5000)
    results.append(test_linear_regression(model, X, Y))

expected = [True, False, False]
print("Results:", results)
print("Expected:", expected)
print("All match:", results == expected)

##### Train the model on a tiny 1D example

Fit the SimpleLinearRegression model on a toy dataset and visualize the fitted line.

Use the fitted model to predict the score for studying 1.5 and 3.5 hours.

In [ ]:
# Given dataset
X = np.array([1, 2, 3, 4, 5, 6, 7, 8])  # Hours studied
Y = np.array([50, 60, 70, 80, 85, 90, 95, 100])  # Test score

X_test = np.array([1.5, 3.5])

model = SimpleLinearRegression(learning_rate=0.01, iterations=1000)
model.fit(X, Y)
model.plot_regression_line(X, Y)

predicted1 = model.predict(1.5)
predicted2 = model.predict(3.5)
print("Predicted score for 1.5 hours:", predicted1)
print("Predicted score for 3.5 hours:", predicted2)

# Part 3: Can a regression-style model separate classes?

A straight-line decision boundary g(x) = mx + b can be used to classify 2D points by thresholding above/below the line. Below we construct a simple example to visualize this idea.

Consider the following plot separating two noisy linear trends.

In [ ]:
x_axis = np.linspace(0, 1, 100)
class_a = 2 * x_axis + 1 + np.random.normal(1, 0.2, x_axis.shape[0])
class_b = 2 * x_axis + 1 + np.random.normal(-1, 0.2, x_axis.shape[0])

plt.scatter(x_axis, class_a, color="r", label="Class A")
plt.scatter(x_axis, class_b, color="b", label="Class B")
plt.legend()

There is an evident linear separator that we could draw by hand. Next, we’ll let the computer search for one.

Define a classifier via a line: g(x; θ) = θ1 x + θ2. We’ll predict class A if y > g(x; θ), else class B. Gradient descent can tune θ on a suitable loss.

We first start with a randomly initialized line; it will likely be inaccurate. Then we’ll optimize θ via gradient descent to reduce mean squared error between y and g(x; θ).

Create a reproducible synthetic dataset with a true linear separator, then color points by their true class relative to that separator.

In [ ]:
m_star = 0.6
b_star = 0.05

x_data_full = np.linspace(0, 1, 1000)
y_data_full = (
    m_star * x_data_full + b_star + np.random.normal(0, 0.05, x_data_full.shape[0])
)
plt.scatter(x_data_full, y_data_full)
plt.title("Raw")
plt.show()


def calc(theta, x_data):
    return theta[0] * x_data + theta[1]


calced_values = calc([m_star, b_star], x_data_full)
true_x_a = []
true_x_b = []
true_y_a = []
true_y_b = []
for i in range(x_data_full.shape[0]):
    val = y_data_full[i]
    if val > calced_values[i]:
        true_x_a.append(x_data_full[i])
        true_y_a.append(y_data_full[i])
    else:
        true_x_b.append(x_data_full[i])
        true_y_b.append(y_data_full[i])

plt.scatter(true_x_a, true_y_a, color="g")
plt.scatter(true_x_b, true_y_b, color="r")
plt.title("Real Data Split by Class")
plt.show()

Initialize the parameters θ = [θ1, θ2] randomly to define an initial line.

In [ ]:
theta = np.random.rand(2)
print(theta)

Plot the line defined by θ and color points by their classification relative to that line (above = class A, below = class B).

In [ ]:
def calc(el, x_data):
    return el[0] * x_data + el[1]


def separate(x_data, y_data, el):
    calced_values = calc(el, x_data)
    true_x_a, true_x_b, true_y_a, true_y_b = [], [], [], []
    for i in range(x_data.shape[0]):
        val = y_data[i]
        if val > calced_values[i]:
            true_x_a.append(x_data[i])
            true_y_a.append(y_data[i])
        else:
            true_x_b.append(x_data[i])
            true_y_b.append(y_data[i])
    return (true_x_a, true_y_a), (true_x_b, true_y_b)

# Use the previously generated x_data_full, y_data_full, and theta
class_a_points, class_b_points = separate(x_data_full, y_data_full, theta)

# Plot the line defined by g(x; theta)
plt.plot(x_data_full, calc(theta, x_data_full), color="black", label="Decision boundary")
# Color the points based on classification
plt.scatter(*class_a_points, color="g", label="Class A (above line)")
plt.scatter(*class_b_points, color="r", label="Class B (below line)")
plt.legend()
plt.title("Classification using a linear separator")
plt.xlabel("X")
plt.ylabel("Y")
plt.show()

The random initialization is typically inaccurate. Next, we’ll optimize θ via gradient descent using MSE to nudge the line toward the separating boundary.

#### Define the MSE loss

In [ ]:
def mse_loss(true_values, predictions):
    n = len(true_values)
    return np.sum((true_values - predictions) ** 2) / n

#### Train with gradient descent and visualize the resulting separator

We’ll run a short training loop, print periodic loss snapshots, and then plot the learned line and resulting classification.

In [ ]:
lr = 0.01  # learning rate
num_steps = 1000  # number of training steps

theta = np.random.rand(2)
for step in range(num_steps):
    predictions = calc(theta, x_data_full)
    loss = mse_loss(y_data_full, predictions)
    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss}")

    grad_theta1 = -2 * np.mean(x_data_full * (y_data_full - predictions))
    grad_theta2 = -2 * np.mean(y_data_full - predictions)

    theta[0] -= lr * grad_theta1
    theta[1] -= lr * grad_theta2

print("Final Loss:", loss)

class_a_points, class_b_points = separate(x_data_full, y_data_full, theta)

plt.plot(x_data_full, calc(theta, x_data_full), color="black", label="Decision boundary")
plt.scatter(*class_a_points, color="g", label="Class A (above line)")
plt.scatter(*class_b_points, color="r", label="Class B (below line)")
plt.legend()
plt.title("Classification using a linear separator (trained)")
plt.xlabel("X")
plt.ylabel("Y")
plt.show()

#### Why can this approach fail?
Linear regression optimizes a squared-error objective for a continuous target. For binary labels, squared error is a poor surrogate, and the decision boundary that minimizes MSE is not guaranteed to maximize classification accuracy. Noise and non-linear structure further degrade performance. For classification, logistic regression or margin-based methods (e.g., SVMs) are typically better.

In short: using regression for classification is often suboptimal because the loss and link function do not align with the discrete nature of the labels. Linear separators also fail on data that are not linearly separable in the original feature space.

### Higher-dimensional data

A linear decision boundary extends directly to higher-dimensional feature spaces. Each additional feature introduces a corresponding linear term and partial derivative.

For input data with dimensionality $d$, the model has one weight for each feature and a single constant term. The intercept contributions can be combined into this constant, yielding $d+1$ tunable weights.

What happens when the data are not linearly separable?

![Example of nonlinearly separable data](images/nonlinear-data.png)

If the classes are not linearly separable in the original feature space, a straight-line decision boundary will not work well.

Consider a non-linear relationship that becomes linearly separable after a simple transformation.

In [ ]:
y_data_full = x_data_full**2

class_a = x_data_full**2 + np.random.normal(0.5, 0.1, x_data_full.shape[0])
class_b = x_data_full**2 + np.random.normal(0, 0.1, x_data_full.shape[0])

plt.scatter(x_data_full, class_a, color="g")
plt.scatter(x_data_full, class_b, color="r")

A linear separator cannot separate these classes in (x, y) space. But if we transform x → x^2, the classes may become separable in the transformed space.

In [ ]:
x_squared = x_data_full**2

plt.scatter(x_squared, class_a, color="g")
plt.scatter(x_squared, class_b, color="r")

plt.xlabel("x squared")

By mapping inputs into a higher-dimensional feature space (a kernel mapping), a linear separator can succeed where it fails in the original space. For complex data, different kernel functions provide different feature mappings.

#### Examples of common kernels

- Polynomial kernel
- Gaussian (RBF) kernel
- Laplacian kernel

Choosing a good fixed feature mapping (kernel) is hard. Neural networks instead learn the representation (feature space) directly from data, which is one reason they are so powerful.

### Appendix: Optional PyTorch neural network demo (California Housing)

This section is optional and requires PyTorch, which is a relatively large dependency. To keep the notebook lightweight by default, it is gated by a flag. Set RUN_TORCH_DEMO = True in the next cell to run.

If PyTorch is not installed, the demo cell will print an instruction message instead of failing. To install a CPU-only build of PyTorch:
- pip install torch --index-url https://download.pytorch.org/whl/cpu

If you have a compatible GPU/CUDA setup, consult the official PyTorch installation selector for the appropriate command.

In [ ]:
# Toggle to run the PyTorch demo
RUN_TORCH_DEMO = False

if RUN_TORCH_DEMO:
    try:
        import torch  # Check availability first
    except ImportError:
        print(
            "PyTorch is not installed. To run this demo, install it first, e.g.:\n"
            "  pip install torch --index-url https://download.pytorch.org/whl/cpu"
        )
    else:
        from sklearn.datasets import fetch_california_housing
        from sklearn.model_selection import train_test_split
        from torch.utils.data import DataLoader, TensorDataset
        import torch.nn as nn
        import torch.optim as optim
        import matplotlib.pyplot as plt

        # Reproducibility
        torch.manual_seed(42)
        np.random.seed(42)

        # Load the data
        data = fetch_california_housing()
        X, y = data.data, data.target

        # Split the data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # Tensors and dataloader
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

        class RegressionModel(nn.Module):
            def __init__(self, input_size):
                super().__init__()
                self.fc1 = nn.Linear(input_size, 64)
                self.fc2 = nn.Linear(64, 32)
                self.fc3 = nn.Linear(32, 1)
            def forward(self, x):
                x = torch.relu(self.fc1(x))
                x = torch.relu(self.fc2(x))
                x = self.fc3(x)
                return x

        model = RegressionModel(X_train.shape[1])
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        num_epochs = 50
        train_losses = []

        for epoch in range(num_epochs):
            running_loss = 0.0
            for inputs, targets in train_dataloader:
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs.squeeze(), targets)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * inputs.size(0)
            epoch_loss = running_loss / len(train_dataset)
            train_losses.append(epoch_loss)
            print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

        plt.plot(train_losses, label="Training Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Training Loss Curve")
        plt.legend()
        plt.show()

        with torch.no_grad():
            outputs = model(X_test_tensor)
            test_loss = criterion(outputs.squeeze(), y_test_tensor)
            print(f"Test Loss: {test_loss.item():.4f}")
else:
    print("Skipping PyTorch demo. Set RUN_TORCH_DEMO = True to run (requires torch).")

## Appendix: Closed-form solution for linear regression

The normal equations give the exact solution for θ that minimizes MSE in ordinary least squares.

Given L = (1/N) (Xθ − y)^T (Xθ − y), differentiate with respect to θ and set the gradient to zero.

This yields the normal equations X^T X θ = X^T y.

Solution:

θ = (X^T X)^{-1} X^T y